In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND CONFIGURATION
# ===================================================

from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import Window
from pyspark.sql import functions as F


"""
Transform Bronze production-lot records into validated Silver records
and isolate rejected records in the quarantine layer.

The transformation applies strict data types, deterministic business-key
deduplication, record-level quality controls, and device-reference checks.
"""

SOURCE_TABLE = "semiconplus_portfolio.bronze.production_lots"
SILVER_TABLE = "semiconplus_portfolio.silver.production_lots"
QUARANTINE_TABLE = "semiconplus_portfolio.quarantine.production_lots"

# Device reference data generated with the five-year source dataset.
DEVICE_REFERENCE_PATH = (
    "/Volumes/semiconplus_portfolio/"
    "landing/external_source/reference/devices.csv"
)

EXPECTED_BRONZE_ROW_COUNT = 18_126
EXPECTED_DISTINCT_LOT_COUNT = 18_125

PIPELINE_RUN_ID = str(uuid4())
PIPELINE_START_TIME = datetime.now(timezone.utc)

print(f"Pipeline run ID: {PIPELINE_RUN_ID}")
print(f"Pipeline start UTC: {PIPELINE_START_TIME.isoformat()}")
print(f"Bronze source: {SOURCE_TABLE}")
print(f"Silver target: {SILVER_TABLE}")
print(f"Quarantine target: {QUARANTINE_TABLE}")

In [0]:
# ===================================================
# BLOCK 2 — DEPENDENCY CHECKS
# ===================================================

"""
Confirm that the required Bronze table and device-reference source are
available before starting the Silver transformation.
"""

assert spark.catalog.tableExists(SOURCE_TABLE), (
    f"Required Bronze table does not exist: {SOURCE_TABLE}"
)

reference_files = dbutils.fs.ls(
    "/Volumes/semiconplus_portfolio/landing/external_source/reference"
)

reference_file_names = {file_info.name for file_info in reference_files}

assert "devices.csv" in reference_file_names, (
    "Required device-reference file was not found: devices.csv"
)

print("Silver pipeline dependencies are available.")

In [0]:
# ===================================================
# BLOCK 3 — LOAD AND RECONCILE BRONZE
# ===================================================

"""
Load the Bronze dataset and enforce the approved initial-load control
totals before applying downstream transformations.
"""

bronze_df = spark.table(SOURCE_TABLE)

bronze_row_count = bronze_df.count()
bronze_distinct_lot_count = bronze_df.select("lot_id").distinct().count()

print(f"Bronze rows: {bronze_row_count:,}")
print(f"Distinct Bronze lot IDs: {bronze_distinct_lot_count:,}")

assert bronze_row_count == EXPECTED_BRONZE_ROW_COUNT, (
    f"Expected {EXPECTED_BRONZE_ROW_COUNT:,} Bronze rows, "
    f"but found {bronze_row_count:,}."
)

assert bronze_distinct_lot_count == EXPECTED_DISTINCT_LOT_COUNT, (
    f"Expected {EXPECTED_DISTINCT_LOT_COUNT:,} distinct lot IDs, "
    f"but found {bronze_distinct_lot_count:,}."
)

In [0]:
# ===================================================
# BLOCK 4 — LOAD DEVICE REFERENCE DATA
# ===================================================

"""
Load the approved device master used to validate production-lot device
identifiers.

Only the device business key is required by this transformation. Other
reference attributes will be modeled separately when reference tables
are promoted into the Silver layer.
"""

device_reference_raw_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(DEVICE_REFERENCE_PATH)
)

assert "device_id" in device_reference_raw_df.columns, (
    "The device-reference file does not contain device_id. "
    f"Columns found: {device_reference_raw_df.columns}"
)

valid_devices_df = (
    device_reference_raw_df
    .select(F.trim(F.col("device_id")).alias("reference_device_id"))
    .filter(F.col("reference_device_id").isNotNull())
    .filter(F.col("reference_device_id") != "")
    .dropDuplicates(["reference_device_id"])
)

device_reference_count = valid_devices_df.count()

print(f"Validated device-reference keys: {device_reference_count:,}")

assert device_reference_count == 30, (
    f"Expected 30 device-reference keys, found {device_reference_count}."
)

assert (
    valid_devices_df
    .groupBy("reference_device_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
    == 0
), "Duplicate keys remain in the validated device reference."

In [0]:
# ===================================================
# BLOCK 5 — STANDARDIZE AND TYPE SOURCE FIELDS
# ===================================================

"""
Standardize text fields and convert source values to their approved
business data types.

try_cast returns null for invalid values so affected records can be
routed to quarantine without terminating the complete pipeline run.
"""

typed_df = bronze_df.select(
    F.trim(F.col("lot_id")).alias("lot_id"),
    F.expr("try_cast(trim(production_date) AS date)").alias(
        "production_date"
    ),
    F.trim(F.col("device_id")).alias("device_id"),
    F.trim(F.col("product_group_id")).alias("product_group_id"),
    F.trim(F.col("site_id")).alias("site_id"),
    F.trim(F.col("equipment_id")).alias("equipment_id"),
    F.expr("try_cast(trim(start_timestamp_utc) AS timestamp)").alias(
        "start_timestamp_utc"
    ),
    F.expr("try_cast(trim(quantity_started) AS bigint)").alias(
        "quantity_started"
    ),
    F.expr("try_cast(trim(quantity_passed) AS bigint)").alias(
        "quantity_passed"
    ),
    F.expr("try_cast(trim(quantity_failed) AS bigint)").alias(
        "quantity_failed"
    ),
    F.expr("try_cast(trim(actual_yield) AS decimal(9,6))").alias(
        "source_actual_yield"
    ),
    F.trim(F.col("test_program_revision")).alias(
        "test_program_revision"
    ),
    F.trim(F.col("source_system")).alias("source_system"),
    F.col("_source_file_path"),
    F.col("_source_file_name"),
    F.col("_source_file_modification_time"),
    F.col("_ingested_at_utc"),
    F.col("_pipeline_run_id").alias("_bronze_pipeline_run_id"),
    F.col("_rescued_data"),
)

print("Source standardization and type conversion completed.")

In [0]:
# ===================================================
# BLOCK 6 — APPLY DETERMINISTIC DEDUPLICATION
# ===================================================

"""
Rank records within each production-lot business key.

The latest source-file modification time is retained. File path and
ingestion timestamp provide stable tie-breakers when duplicate records
share the same modification timestamp.
"""

deduplication_window = (
    Window
    .partitionBy("lot_id")
    .orderBy(
        F.col("_source_file_modification_time").desc_nulls_last(),
        F.col("_source_file_path").desc_nulls_last(),
        F.col("_ingested_at_utc").desc_nulls_last(),
    )
)

ranked_df = typed_df.withColumn(
    "_business_key_rank",
    F.row_number().over(deduplication_window),
)

duplicate_record_count = ranked_df.filter(
    F.col("_business_key_rank") > 1
).count()

print(f"Duplicate records identified: {duplicate_record_count:,}")

assert duplicate_record_count == 1, (
    f"Expected one duplicate record, found {duplicate_record_count}."
)

In [0]:
# ===================================================
# BLOCK 7 — APPLY DEVICE-REFERENCE CHECK
# ===================================================

"""
Join the ranked production records to the approved device reference and
retain an explicit match indicator for quality-rule evaluation.
"""

reference_checked_df = (
    ranked_df
    .join(
        F.broadcast(valid_devices_df),
        ranked_df.device_id == valid_devices_df.reference_device_id,
        "left",
    )
    .withColumn(
        "_device_reference_valid",
        F.col("reference_device_id").isNotNull(),
    )
    .drop("reference_device_id")
)

unknown_device_count = reference_checked_df.filter(
    ~F.col("_device_reference_valid")
).count()

print(f"Unknown-device records identified: {unknown_device_count:,}")

assert unknown_device_count == 1, (
    f"Expected one unknown-device record, found {unknown_device_count}."
)

In [0]:
# ===================================================
# BLOCK 8 — ASSIGN DATA-QUALITY REASONS
# ===================================================
"""
Evaluate record-level quality rules and attach all applicable rejection
reasons to each source record.

An array is used so one rejected record can retain multiple failure
reasons without losing diagnostic detail.
"""

quality_checked_df = reference_checked_df.withColumn(
    "_quality_reasons",
    F.array_compact(
        F.array(
            F.when(
                F.col("lot_id").isNull() | (F.col("lot_id") == ""),
                F.lit("MISSING_LOT_ID"),
            ),
            F.when(
                F.col("_business_key_rank") > 1,
                F.lit("DUPLICATE_LOT_ID"),
            ),
            F.when(
                F.col("production_date").isNull(),
                F.lit("INVALID_PRODUCTION_DATE"),
            ),
            F.when(
                F.col("start_timestamp_utc").isNull(),
                F.lit("INVALID_START_TIMESTAMP"),
            ),
            F.when(
                F.col("device_id").isNull() | (F.col("device_id") == ""),
                F.lit("MISSING_DEVICE_ID"),
            ),
            F.when(
                ~F.col("_device_reference_valid"),
                F.lit("UNKNOWN_DEVICE_ID"),
            ),
            F.when(
                F.col("product_group_id").isNull()
                | (F.col("product_group_id") == ""),
                F.lit("MISSING_PRODUCT_GROUP_ID"),
            ),
            F.when(
                F.col("site_id").isNull() | (F.col("site_id") == ""),
                F.lit("MISSING_SITE_ID"),
            ),
            F.when(
                F.col("equipment_id").isNull()
                | (F.col("equipment_id") == ""),
                F.lit("MISSING_EQUIPMENT_ID"),
            ),
            F.when(
                F.col("quantity_started").isNull()
                | (F.col("quantity_started") <= 0),
                F.lit("INVALID_QUANTITY_STARTED"),
            ),
            F.when(
                F.col("quantity_passed").isNull()
                | (F.col("quantity_passed") < 0),
                F.lit("INVALID_QUANTITY_PASSED"),
            ),
            F.when(
                F.col("quantity_failed").isNull()
                | (F.col("quantity_failed") < 0),
                F.lit("INVALID_QUANTITY_FAILED"),
            ),
            F.when(
                F.col("quantity_passed") + F.col("quantity_failed")
                != F.col("quantity_started"),
                F.lit("QUANTITY_RECONCILIATION_FAILED"),
            ),
            F.when(
                F.col("source_actual_yield").isNull()
                | (F.col("source_actual_yield") < F.lit(0))
                | (F.col("source_actual_yield") > F.lit(1)),
                F.lit("INVALID_ACTUAL_YIELD"),
            ),
            F.when(
                F.col("_rescued_data").isNotNull(),
                F.lit("UNEXPECTED_SOURCE_FIELDS"),
            ),
        )
    ),
)

quality_checked_df = quality_checked_df.withColumn(
    "_quality_status",
    F.when(
        F.size(F.col("_quality_reasons")) == 0,
        F.lit("ACCEPTED"),
    ).otherwise(F.lit("REJECTED")),
)

quality_summary_df = (
    quality_checked_df
    .groupBy("_quality_status")
    .count()
    .orderBy("_quality_status")
)

display(quality_summary_df)

In [0]:
# ===================================================
# BLOCK 9 — BUILD ACCEPTED SILVER DATASET
# ===================================================

"""
Build the accepted Silver dataset with approved data types and a
recalculated yield measure derived from validated quantities.

The source yield remains available for comparison while calculated yield
becomes the controlled analytical measure.
"""

silver_df = (
    quality_checked_df
    .filter(F.col("_quality_status") == "ACCEPTED")
    .select(
        "lot_id",
        "production_date",
        "device_id",
        "product_group_id",
        "site_id",
        "equipment_id",
        "start_timestamp_utc",
        "quantity_started",
        "quantity_passed",
        "quantity_failed",
        "source_actual_yield",
        (
            F.col("quantity_passed") / F.col("quantity_started")
        ).cast("decimal(9,6)").alias("calculated_yield"),
        "test_program_revision",
        "source_system",
        "_source_file_path",
        "_source_file_name",
        "_source_file_modification_time",
        "_ingested_at_utc",
        "_bronze_pipeline_run_id",
        F.lit(PIPELINE_RUN_ID).alias("_silver_pipeline_run_id"),
        F.current_timestamp().alias("_silver_processed_at_utc"),
    )
)

silver_row_count = silver_df.count()

print(f"Accepted Silver rows: {silver_row_count:,}")

In [0]:
# ===================================================
# BLOCK 10 — BUILD QUARANTINE DATASET
# ===================================================


"""
Build the quarantine dataset with rejected values, complete quality
reasons, source lineage, and pipeline execution metadata.
"""

quarantine_df = (
    quality_checked_df
    .filter(F.col("_quality_status") == "REJECTED")
    .select(
        "lot_id",
        "production_date",
        "device_id",
        "product_group_id",
        "site_id",
        "equipment_id",
        "start_timestamp_utc",
        "quantity_started",
        "quantity_passed",
        "quantity_failed",
        "source_actual_yield",
        "test_program_revision",
        "source_system",
        "_business_key_rank",
        "_device_reference_valid",
        "_quality_reasons",
        "_source_file_path",
        "_source_file_name",
        "_source_file_modification_time",
        "_ingested_at_utc",
        "_bronze_pipeline_run_id",
        "_rescued_data",
        F.lit(PIPELINE_RUN_ID).alias("_silver_pipeline_run_id"),
        F.current_timestamp().alias("_quarantined_at_utc"),
    )
)

quarantine_row_count = quarantine_df.count()

print(f"Quarantined rows: {quarantine_row_count:,}")


In [0]:
# ===================================================
# BLOCK 11 — PRE-WRITE RECONCILIATION
# ===================================================

"""
Reconcile accepted and quarantined records to the complete Bronze input
before replacing the current Silver outputs.
"""

assert silver_row_count + quarantine_row_count == bronze_row_count, (
    "Silver reconciliation failed: accepted and quarantined records do "
    "not equal the Bronze source count."
)

assert quarantine_row_count >= 2, (
    "Expected at least the deliberate duplicate and unknown-device "
    "records in quarantine."
)

assert silver_df.select("lot_id").distinct().count() == silver_row_count, (
    "Accepted Silver records are not unique by lot_id."
)

print("Pre-write Silver reconciliation passed.")

In [0]:
# ===================================================
# BLOCK 12 — WRITE SILVER AND QUARANTINE TABLES
# ===================================================

"""
Replace the current Silver and quarantine snapshots atomically at the
individual table level.

Overwrite mode makes development reruns repeatable and prevents duplicate
records when the same validated Bronze snapshot is processed again.
"""

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

(
    quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"Silver table written: {SILVER_TABLE}")
print(f"Quarantine table written: {QUARANTINE_TABLE}")


In [0]:
# ===================================================
# BLOCK 13 — POST-WRITE CONTROLS
# ===================================================

"""
Confirm that persisted output counts match the validated in-memory
datasets and still reconcile to the Bronze source.
"""

persisted_silver_count = spark.table(SILVER_TABLE).count()
persisted_quarantine_count = spark.table(QUARANTINE_TABLE).count()

print(f"Persisted Silver rows: {persisted_silver_count:,}")
print(f"Persisted quarantine rows: {persisted_quarantine_count:,}")

assert persisted_silver_count == silver_row_count
assert persisted_quarantine_count == quarantine_row_count
assert persisted_silver_count + persisted_quarantine_count == bronze_row_count

PIPELINE_END_TIME = datetime.now(timezone.utc)
PIPELINE_DURATION_SECONDS = (
    PIPELINE_END_TIME - PIPELINE_START_TIME
).total_seconds()

print("SILVER TRANSFORMATION PASSED")
print(f"Pipeline end UTC: {PIPELINE_END_TIME.isoformat()}")
print(f"Duration: {PIPELINE_DURATION_SECONDS:.2f} seconds")